# **Проверка оценки CATE: BLP, GATES, CLAN и RATE**

Практика к модулю [«Гетерогенные эффекты»](https://ai-interpretability.school), урок «Как читать оценку и где ошибиться», шаги «Есть ли неоднородность вообще: BLP, GATES и CLAN» и «RATE и кривая TOC: кого обслужить первым».

Калибровка по квартилям и коэффициент Qini из тетради «Гетерогенные эффекты» показывают, ранжирует ли модель объекты по эффекту, но без интервалов и без проверки гипотезы. Здесь мы соберем руками формальные проверки:

1. BLP ([Chernozhukov, Demirer, Duflo и Fernández-Val, 2025](https://arxiv.org/abs/1712.04802)): есть ли неоднородность и несет ли ее прогноз модели;
2. GATES: средний эффект в пяти группах по прогнозу модели, с интервалами;
3. CLAN: чем отличаются объекты групп с наименьшим и наибольшим эффектом;
4. RATE и кривая TOC ([Yadlowsky et al., 2025](https://arxiv.org/abs/2111.07966)): насколько хорошо прогноз модели выбирает, кого обслужить первым;
5. где ломается: эффект без неоднородности и проверка на тех же данных, где учили модель;
6. библиотечный вызов `econml.validate.DRTester` и чем он отличается от статей.

Все проверки устроены одинаково: модель эффекта учится на одной части выборки (вспомогательной), а проверяется на другой (основной), где ее прогноз — просто признак, фиксированный заранее.

Тетрадь идет на процессоре; весь прогон занимает около минуты.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from econml.dml import CausalForestDML
from econml.validate import DRTester

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # ложные предупреждения numpy на некоторых процессорах
SEED = 0
P = 0.5                                                                 # вероятность воздействия в эксперименте
pd.set_option('display.width', 140)

## 1. Данные, прогноз модели и дважды робастные оценки

Рандомизированный эксперимент: 4000 объектов, пять признаков $X\sim\mathcal N(0, I)$, воздействие у случайной половины ($p=0.5$). Эффект $\tau(x)=1+1.5\,x_0$, исход без воздействия $2x_1+x_2^2$, шум со стандартным отклонением 2 — шумнее, чем в тетради «Гетерогенные эффекты», чтобы интервалы были заметны.

Первые 2000 объектов — вспомогательная часть $A$: на ней учим причинный лес, его прогноз $S(x)$ — «прокси» эффекта в терминах статьи, и модель исхода без воздействия $B(x)$. Вторые 2000 — основная часть $M$: на ней все проверки. Для RATE нужны еще дважды робастные оценки эффекта $\hat\Gamma_i$ на основной части, с моделями исхода из других фолдов основной части; склонность известна, $p=0.5$.

In [ ]:
def simulate(n, r, heterogeneous=True):
    Xs = r.normal(size=(n, 5))
    ts = 1 + 1.5 * Xs[:, 0] if heterogeneous else np.ones(n)
    Ts = r.binomial(1, P, n)
    Ys = 2 * Xs[:, 1] + Xs[:, 2] ** 2 + Ts * ts + 2 * r.normal(size=n)
    return Xs, Ts, Ys, ts


def gb():
    return GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED)


def proxies(Xs, Ts, Ys, fit, ev):
    # лес и модель исхода учатся на fit, прогноз S и B считается на ev
    cf = CausalForestDML(model_y=gb(), model_t=LogisticRegression(), discrete_treatment=True,
                         n_estimators=500, min_samples_leaf=20, random_state=SEED).fit(Ys[fit], Ts[fit], X=Xs[fit])
    base = gb().fit(Xs[fit][Ts[fit] == 0], Ys[fit][Ts[fit] == 0])
    return cf, cf.effect(Xs[ev]), base.predict(Xs[ev])


def aipw_scores(Xa, Ta, Ya):
    G = np.zeros(len(Ya))
    for a, b in KFold(5, shuffle=True, random_state=SEED).split(Xa):
        m1 = gb().fit(Xa[a][Ta[a] == 1], Ya[a][Ta[a] == 1]).predict(Xa[b])
        m0 = gb().fit(Xa[a][Ta[a] == 0], Ya[a][Ta[a] == 0]).predict(Xa[b])
        G[b] = m1 - m0 + Ta[b] * (Ya[b] - m1) / P - (1 - Ta[b]) * (Ya[b] - m0) / (1 - P)
    return G


X, T, Y, tau = simulate(4000, np.random.default_rng(SEED))
A, M = np.arange(2000), np.arange(2000, 4000)
cf, S, B = proxies(X, T, Y, A, M)
G = aipw_scores(X[M], T[M], Y[M])
print('истинный средний эффект на M:', round(tau[M].mean(), 3), '| среднее Gamma:', round(G.mean(), 3))
print('корреляция прогноза S с истинным эффектом на M:', round(np.corrcoef(S, tau[M])[0, 1], 3))

## 2. BLP: есть ли неоднородность

Лучший линейный предиктор эффекта по прогнозу модели: $\mathrm{BLP}[\tau(X)\mid S]=\beta_1+\beta_2\,(S-\mathbb E S)$, где $\beta_1=\mathbb E\,\tau(X)$ — средний эффект, а $\beta_2=\mathrm{Cov}(\tau(X), S)/\mathrm{Var}(S)$. Если прогноз идеален, $\beta_2=1$; если он чистый шум или неоднородности нет, $\beta_2=0$. Отвергнутая гипотеза $\beta_2=0$ значит сразу две вещи: неоднородность есть и прогноз ее несет.

$\tau(X)$ не наблюдается, но коэффициенты оцениваются взвешенной регрессией исхода на основной части (стратегия A статьи):

$$Y_i=\alpha_0+\alpha_1B(X_i)+\beta_1\,(T_i-p)+\beta_2\,(T_i-p)\,(S_i-\bar S)+\varepsilon_i$$

с весами $1/(p(1-p))$; при постоянной склонности веса одинаковые, и это обычный МНК. Слагаемое $\alpha_1B(X_i)$ уменьшает шум, но не нужно для несмещенности: множитель $(T_i-p)$ делает регрессоры $\beta_1$ и $\beta_2$ ортогональными любой функции признаков.

**Задание 1. Напишите функцию `blp(Ym, Tm, S, B)`, которая строит эту регрессию (`sm.WLS(..., weights=...).fit(cov_type='HC1')`) и возвращает оценки $\beta_1$, $\beta_2$ и их стандартные ошибки. Посчитайте их на основной части. Чему равен $\beta_2$ и отвергается ли гипотеза $\beta_2=0$?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 3. GATES: средний эффект в группах по прогнозу

Разобьем основную часть на $K=5$ групп по квантилям прогноза $S$: $G_1$ — пятая часть с наименьшим прогнозом, $G_5$ — с наибольшим. GATES — средний эффект в каждой группе, $\gamma_k=\mathbb E[\tau(X)\mid G_k]$. Его оценивает та же регрессия, только вместо одного слагаемого с $(T_i-p)$ — по одному на группу:

$$Y_i=\alpha_0+\alpha_1B(X_i)+\sum_{k=1}^K\gamma_k\,(T_i-p)\,\mathbb 1[i\in G_k]+\varepsilon_i.$$

Это калибровка по квантилям из урока, но с интервалами и с поправкой на шум исхода. Если модель ранжирует эффект осмысленно, $\gamma_k$ растут с $k$, а разность $\gamma_K-\gamma_1$ значимо больше нуля.

**Задание 2. Напишите функцию `gates(Ym, Tm, S, B, K=5)`, которая возвращает оценки $\gamma_1,\dots,\gamma_K$, их стандартные ошибки и номера групп объектов (группы — `pd.qcut(S, K, labels=False)`). Выведите таблицу: $\gamma_k$, стандартная ошибка и средний истинный эффект в группе. Чему равна разность $\gamma_5-\gamma_1$?**

In [ ]:
# Ваш код здесь

## 4. CLAN: чем отличаются крайние группы

Когда GATES показывает неоднородность, естественный вопрос — кто эти объекты. CLAN сравнивает средние признаков в группе с наименьшим эффектом $G_1$ и с наибольшим $G_K$: $\delta_1=\mathbb E[g(X)\mid G_1]$, $\delta_K=\mathbb E[g(X)\mid G_K]$. Это просто средние по наблюдаемым признакам, интервал для разности — обычный двухвыборочный.

**Задание 3. Для каждого из пяти признаков выведите среднее в $G_1$, среднее в $G_5$, их разность и ее стандартную ошибку $\sqrt{s_1^2/n_1+s_5^2/n_5}$. По какому признаку крайние группы различаются сильнее всего и на сколько?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 5. RATE и кривая TOC

BLP и GATES спрашивают, есть ли неоднородность. RATE спрашивает о другом: если обслуживать объекты в порядке убывания прогноза $S$, насколько эффект у первых больше среднего. Кривая TOC (targeting operator characteristic):

$$\mathrm{TOC}(u)=\mathbb E\bigl[\tau(X)\mid S\text{ в верхней доле }u\bigr]-\mathbb E\,\tau(X),\qquad u\in(0,1].$$

$\mathrm{TOC}(1)=0$; у прогноза, не связанного с эффектом, кривая около нуля везде. RATE — взвешенная площадь под кривой: $\theta_\alpha=\int_0^1\alpha(u)\,\mathrm{TOC}(u)\,du$. При $\alpha(u)=1$ это AUTOC, при $\alpha(u)=u$ — коэффициент Qini в нормировке Yadlowsky et al. Оценка подставляет вместо $\tau(X_i)$ дважды робастные оценки $\hat\Gamma_i$: $\widehat{\mathrm{TOC}}(j/n)=\frac1j\sum_{k\le j}\hat\Gamma_{(k)}-\bar\Gamma$, где $\hat\Gamma_{(k)}$ — оценка объекта, стоящего $k$-м по убыванию $S$. Стандартную ошибку авторы предлагают считать бутстрепом по половинам выборки: оценку пересчитывают на случайных половинах, и стандартное отклонение по повторам делят на $\sqrt2$.

**Задание 4. Напишите функцию `rate(S, G)`, которая возвращает кривую $\widehat{\mathrm{TOC}}$ в точках $u=j/n$, AUTOC и Qini (средние $\widehat{\mathrm{TOC}}$ и $u\cdot\widehat{\mathrm{TOC}}$ по точкам). Посчитайте их на основной части и стандартные ошибки бутстрепом по 200 случайным половинам (`np.random.default_rng(SEED)`, `choice(n, n // 2, replace=False)`). Нарисуйте кривую TOC. Чему равен AUTOC и отличается ли он от нуля?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 6. Где ломается

### 6.1. Эффект без неоднородности

Те же данные, но эффект у всех одинаковый: $\tau(x)=1$. Хорошая проверка должна сказать, что неоднородности нет, хотя лес все равно выдаст разные прогнозы разным объектам.

### 6.2. Проверка на тех же данных, где училась модель

Все проверки выше опираются на то, что прогноз $S$ получен на другой части выборки. Если посчитать BLP на той же части $A$, где учился лес, шум, под который лес подогнался, войдет в $S$ и в исход одновременно.

**Задание 5. Сгенерируйте данные `simulate(4000, np.random.default_rng(1), heterogeneous=False)`. Посчитайте $\beta_2$ BLP с его стандартной ошибкой, $\gamma_1,\dots,\gamma_5$ GATES и AUTOC со стандартной ошибкой на основной части $M$ (лес и $B$ учатся на $A$). Затем посчитайте $\beta_2$ BLP на самой части $A$: прогнозы `proxies(X0, T0, Y0, A, A)`. Что показывают проверки на $M$ и что — на $A$?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 7. Библиотечный вызов: `econml.validate.DRTester`

В EconML те же проверки собраны в `DRTester`: `fit_nuisance` считает дважды робастные оценки на основной части с моделями, обученными на вспомогательной, `evaluate_blp` — BLP, `evaluate_cal` — калибровку по группам, `evaluate_uplift` — Qini или TOC с бутстрепом. Устроены они не так, как в статьях, и это видно в числах:

- `evaluate_blp` — регрессия МНК дважды робастных оценок на прогноз без центрирования, а не взвешенная регрессия исхода из статьи Chernozhukov et al.; наклон оценивает то же $\beta_2$, но с другими вспомогательными моделями;
- `evaluate_uplift` интегрирует кривую только по процентилям прогноза от 5 до 95 (пороги — процентили прогноза на вспомогательной части), а не по всей доле от 0 до 1; с `metric='toc'` под интегралом $\mathrm{TOC}(u)$, и AUTOC библиотеки меньше нашего, с `metric='qini'` — $u\cdot\mathrm{TOC}(u)$, который на краях мал, и число почти совпадает с нашим Qini;
- стандартные ошибки `evaluate_uplift` считаются через функцию влияния, а бутстреп `n_bootstrap` нужен только для равномерной доверительной полосы кривой.

**Задание 6. Создайте `DRTester(model_regression=gb(), model_propensity=LogisticRegression(), cate=cf)`, вызовите `fit_nuisance(X[M], T[M], Y[M], X[A], T[A], Y[A])` и выведите наклон `evaluate_blp(X[M], X[A])`, `cal_r_squared` из `evaluate_cal()` и коэффициенты `evaluate_uplift(metric='qini', n_bootstrap=200)` и `evaluate_uplift(metric='toc', n_bootstrap=200)` со стандартными ошибками. Какое число библиотеки совпадает с нашим до третьего знака?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## Реализации

- `econml.validate.DRTester` (EconML, лицензия MIT) — BLP, калибровка и Qini/TOC, как в разделе 7.
- R-пакет [grf](https://github.com/grf-labs/grf) (лицензия GPL-3.0): `test_calibration()` — проверка в духе BLP с коэффициентами при среднем и при отклонении прогноза леса, `rank_average_treatment_effect()` — RATE с AUTOC и Qini, авторская реализация Yadlowsky et al.
- R-пакет [GenericML](https://github.com/mwelz/GenericML), на который ссылаются Chernozhukov et al., — BLP, GATES и CLAN с повторными разбиениями. Статья делит выборку не один раз, а много (в примере — 250), и сообщает медиану оценок и p-значений по разбиениям: результат одного разбиения зависит от случайного деления. В тетради разбиение одно, чтобы было с чем сверить числа.

## Что унести из тетради

- **BLP отвечает, есть ли неоднородность и несет ли ее прогноз,** одним коэффициентом $\beta_2$ с интервалом.
- **GATES — калибровка по квантилям с интервалами,** CLAN — описание крайних групп по наблюдаемым признакам.
- **RATE оценивает прогноз как правило очередности:** насколько эффект у обслуженных первыми больше среднего.
- **Без неоднородности проверки на отложенной части молчат,** хотя прогнозы леса разные.
- **Проверка на данных, где училась модель, врет:** подогнанный шум выглядит как сильная неоднородность.

Источники:
1. [Chernozhukov, Demirer, Duflo и Fernández-Val, 2025. Generic machine learning inference on heterogeneous treatment effects in randomized experiments](https://arxiv.org/abs/1712.04802)
2. [Yadlowsky, Fleming, Shah, Brunskill и Wager, 2025. Evaluating treatment prioritization rules via rank-weighted average treatment effects](https://arxiv.org/abs/2111.07966)
3. [EconML](https://github.com/py-why/EconML), [grf](https://github.com/grf-labs/grf)